# Semana 03
## Limpieza y preparacion de datos

**Objetivo**: aplicar reglas explícitas de limpieza y dejar un log reproducible para uso analítico y visual.

**Herramientas teoricas de la semana**
- limpieza orientada a visualizacion
- tratamiento de nulos
- homologacion de categorias
- bitacora de transformaciones


### Agenda sugerida de 4 horas
- 0:00 - 0:30: teoria de limpieza y costo epistemico de intervenir
- 0:30 - 1:20: limpieza paso a paso con `pandas`
- 1:20 - 2:15: comparacion antes vs despues
- 2:15 - 3:20: construccion de bitacora y exportables
- 3:20 - 4:00: reflexion metodologica


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _shared import (
    make_base_sales,
    introduce_quality_issues,
    profile_dataframe,
    clean_sales_data,
    build_star_schema,
    save_for_tableau,
    ensure_output_dir,
    contrast_ratio,
    make_high_dimensional_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
WEEK = "week-03"
OUTPUT_DIR = ensure_output_dir(WEEK)

dirty = introduce_quality_issues(make_base_sales(n=2200, seed=7), seed=7)
clean, cleaning_log = clean_sales_data(dirty)
print(dirty.shape, clean.shape)
cleaning_log


### Actividad 1. Comparar calidad antes y despues
Aquí medimos si la limpieza realmente mejoró la interpretabilidad sin destruir estructura útil.


In [ ]:
before_profile = profile_dataframe(dirty).set_index('column')
after_profile = profile_dataframe(clean).set_index('column')
comparison = before_profile[['missing_pct', 'n_unique']].join(
    after_profile[['missing_pct', 'n_unique']], lsuffix='_before', rsuffix='_after'
)
comparison.head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
comparison.sort_values('missing_pct_before', ascending=False).head(8).plot(
    kind='bar', y=['missing_pct_before', 'missing_pct_after'], ax=axes[0], title='Nulos antes vs despues', color=['#9ca3af', '#111827']
)
comparison.sort_values('n_unique_before', ascending=False).head(8).plot(
    kind='bar', y=['n_unique_before', 'n_unique_after'], ax=axes[1], title='Cardinalidad antes vs despues', color=['#9ca3af', '#4b5563']
)
plt.tight_layout()


### Actividad 2. Dejar una bitacora que pueda defenderse
La limpieza no se considera completa hasta que puede explicarse. El log debe documentar:
- campo afectado
- problema observado
- regla aplicada
- impacto esperado


In [ ]:
cleaning_log


In [ ]:
save_for_tableau(clean, WEEK, 'clean_sales')
save_for_tableau(cleaning_log, WEEK, 'cleaning_log')
save_for_tableau(comparison.reset_index(), WEEK, 'cleaning_comparison')


### Uso teorico de herramientas
- `str.strip`, `str.title`, `replace` y `to_datetime` modelan reglas explícitas.
- La comparación antes vs después sirve para discutir si una intervención fue razonable.
- El archivo exportado a Tableau ya puede usarse como fuente base del resto del curso.
